# LightGBM + ECFP with Threshold Tuning & DAG Consistency

Adapted from rf_ecfp_tuned. Key changes:
1. LightGBM instead of RandomForest (one binary classifier per class)
2. Per-class threshold optimization
3. DAG consistency enforcement

In [ ]:
# Compute ECFP fingerprints
mol_transformer = MolFromSmilesTransformer()
ecfp = ECFPFingerprint(fp_size=2048, radius=2, count=True)

X_train = ecfp.transform(mol_transformer.transform(train_df['SMILES'].tolist()))
X_val = ecfp.transform(mol_transformer.transform(val_df['SMILES'].tolist()))
X_test = ecfp.transform(mol_transformer.transform(test_df['SMILES'].tolist()))

y_train = train_df[label_cols].values
y_val = val_df[label_cols].values

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

In [ ]:
# Train one LightGBM binary classifier per class
# LightGBM doesn't natively support multi-output, so we loop over classes

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'n_estimators': 500,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'is_unbalance': True,
    'random_state': 42,
    'n_jobs': -1,
}

models = []
y_val_proba = np.zeros((len(X_val), len(label_cols)))

for i in range(len(label_cols)):
    clf = lgb.LGBMClassifier(**lgb_params)
    clf.fit(X_train, y_train[:, i])
    y_val_proba[:, i] = clf.predict_proba(X_val)[:, 1]
    models.append(clf)
    if (i + 1) % 50 == 0:
        print(f"  Trained {i+1}/{len(label_cols)} classes")

print(f"Val proba shape: {y_val_proba.shape}, range: [{y_val_proba.min():.3f}, {y_val_proba.max():.3f}]")

In [ ]:
# Baseline F1 with default threshold
y_val_pred_default = (y_val_proba >= 0.5).astype(int)
f1_default = np.mean([f1_score(y_val[:, i], y_val_pred_default[:, i], zero_division=0) for i in range(len(label_cols))])
print(f"Baseline Val Macro F1 (threshold=0.5): {f1_default:.4f}")

## Part 1: Per-class Threshold Optimization

In [ ]:
def optimize_threshold(y_true, y_proba, thresholds=np.arange(0.01, 0.99, 0.01)):
    """Find optimal threshold for a single class."""
    best_thresh = 0.5
    best_f1 = 0
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh, best_f1

# Optimize threshold for each class
optimal_thresholds = []
per_class_f1_optimized = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba[:, i])
    optimal_thresholds.append(thresh)
    per_class_f1_optimized.append(f1)

optimal_thresholds = np.array(optimal_thresholds)
print(f"Threshold stats: mean={optimal_thresholds.mean():.3f}, min={optimal_thresholds.min():.3f}, max={optimal_thresholds.max():.3f}")
print(f"Optimized Val Macro F1: {np.mean(per_class_f1_optimized):.4f}")

## Part 2: Parse DAG from OBO file

In [ ]:
def parse_obo(filepath):
    """Parse OBO file to extract parent-child relationships."""
    parents = defaultdict(list)
    children = defaultdict(list)
    
    current_id = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('id: class_'):
                current_id = line.split('id: ')[1]
            elif line.startswith('is_a: class_') and current_id:
                parent_id = line.split('is_a: ')[1].split(' !')[0]
                parents[current_id].append(parent_id)
                children[parent_id].append(current_id)
    
    return dict(parents), dict(children)

parents_map, children_map = parse_obo('../data/chebi_classes.obo')
id_to_idx = {col: i for i, col in enumerate(label_cols)}
print(f"Parsed {len(parents_map)} nodes with parents, {len(children_map)} nodes with children")

In [ ]:
def enforce_dag_consistency(proba, parents_map, id_to_idx):
    """Enforce DAG consistency: P(parent) >= max(P(children))."""
    proba = proba.copy()
    our_ids = set(id_to_idx.keys())
    
    for _ in range(50):
        changed = False
        for child_id, parent_ids in parents_map.items():
            if child_id not in our_ids:
                continue
            child_idx = id_to_idx[child_id]
            
            for parent_id in parent_ids:
                if parent_id not in our_ids:
                    continue
                parent_idx = id_to_idx[parent_id]
                
                mask = proba[:, parent_idx] < proba[:, child_idx]
                if mask.any():
                    proba[mask, parent_idx] = proba[mask, child_idx]
                    changed = True
        
        if not changed:
            break
    
    return proba

# Apply DAG consistency to validation probabilities
y_val_proba_dag = enforce_dag_consistency(y_val_proba, parents_map, id_to_idx)
print(f"DAG consistency applied. Proba range: [{y_val_proba_dag.min():.3f}, {y_val_proba_dag.max():.3f}]")

In [ ]:
# Re-optimize thresholds on DAG-consistent probabilities
optimal_thresholds_dag = []
per_class_f1_dag = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba_dag[:, i])
    optimal_thresholds_dag.append(thresh)
    per_class_f1_dag.append(f1)

optimal_thresholds_dag = np.array(optimal_thresholds_dag)
print(f"DAG Threshold stats: mean={optimal_thresholds_dag.mean():.3f}, min={optimal_thresholds_dag.min():.3f}, max={optimal_thresholds_dag.max():.3f}")
print(f"DAG + Threshold Optimized Val Macro F1: {np.mean(per_class_f1_dag):.4f}")

## Part 3: Generate Test Predictions

In [ ]:
# Retrain on full data (train + val)
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = ecfp.transform(mol_transformer.transform(full_df['SMILES'].tolist()))
y_full = full_df[label_cols].values

models_full = []
y_test_proba = np.zeros((len(X_test), len(label_cols)))

for i in range(len(label_cols)):
    clf = lgb.LGBMClassifier(**lgb_params)
    clf.fit(X_full, y_full[:, i])
    y_test_proba[:, i] = clf.predict_proba(X_test)[:, 1]
    models_full.append(clf)
    if (i + 1) % 50 == 0:
        print(f"  Trained {i+1}/{len(label_cols)} classes")

print(f"Test proba shape: {y_test_proba.shape}")

In [ ]:
# Apply DAG consistency and optimized thresholds
y_test_proba_dag = enforce_dag_consistency(y_test_proba, parents_map, id_to_idx)

y_test_pred = np.zeros_like(y_test_proba_dag, dtype=int)
for i in range(len(label_cols)):
    y_test_pred[:, i] = (y_test_proba_dag[:, i] >= optimal_thresholds_dag[i]).astype(int)

print(f"Test predictions shape: {y_test_pred.shape}")
print(f"Avg predictions per sample: {y_test_pred.sum(axis=1).mean():.1f}")

In [ ]:
# Save submission
submission = pd.DataFrame(y_test_pred, columns=label_cols)
submission.insert(0, 'mol_id', test_df['mol_id'].values)
submission.insert(1, 'SMILES', test_df['SMILES'].values)
submission.to_parquet('../submissions/lgbm_ecfp_tuned.parquet', index=False)
print(f"Saved submission: {submission.shape}")

In [ ]:
# Summary
print("="*50)
print("SUMMARY")
print("="*50)
print(f"Baseline (threshold=0.5):     Val F1 = {f1_default:.4f}")
print(f"+ Threshold tuning:           Val F1 = {np.mean(per_class_f1_optimized):.4f}")
print(f"+ DAG consistency + tuning:   Val F1 = {np.mean(per_class_f1_dag):.4f}")

In [ ]:
placeholder